In [4]:
from phi.agent import Agent
from phi.knowledge.langchain import LangChainKnowledgeBase



In [5]:
from langchain.embeddings import OllamaEmbeddings
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from pathlib import Path

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()
# API_KEY=os.environ['EXA_API_KEY']
# api_key=os.environ['OPENAI_API_KEY']
chroma_db_dir = "./Rag/chroma_db"

def load_vector_store():
    # Define the path to the document
    state_of_the_union = r"state_of_the_union.txt"
    
    # Check if the file exists
    if not os.path.exists(state_of_the_union):
        raise FileNotFoundError(f"File not found: {state_of_the_union}")
    
    # -*- Load the document with explicit encoding
    raw_documents = TextLoader(str(state_of_the_union), encoding="utf-8").load()
    
    # -*- Split it into chunks
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
    documents = text_splitter.split_documents(raw_documents)
    
    # -*- Embed each chunk and load it into the vector store
    embeddings = OllamaEmbeddings(model='nomic-embed-text:latest')
    db = FAISS.from_documents(documents, embeddings)
    
    # Save the FAISS index to the specified directory
    db.save_local(chroma_db_dir)
    
    return db


In [7]:
db = load_vector_store()

# -*- Create a retriever from the vector store
retriever = db.as_retriever()

C:\Users\Asus\AppData\Local\Temp\ipykernel_22332\1799724098.py:24: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model='nomic-embed-text:latest')


In [8]:
# -*- Create a knowledge base from the vector store
knowledge_base = LangChainKnowledgeBase(retriever=retriever)

In [10]:

from phi.model.groq import Groq
from phi.agent import Agent
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=Groq(id="llama-3.1-70b-versatile")
agent = Agent(model=model,knowledge_base=knowledge_base, add_references_to_prompt=True)
agent.print_response("What did the president say about technology?",markdown=True)

c:\Project\GenAI\PhiData\Financial AI analyst\Rag_New\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [14]:
api_key=os.getenv("DEEPSEEK_API_KEY")

In [15]:
api_key

'sk-91c58b0c23884b52b97699b788418e70'